# 09 — ResNet-50 Fine-Tuning

**Best model:** ResNet-50 pretrained (val F1 = 0.9091, test F1 = 0.9004)

**Goals:**
1. Load `final_resnet50_pretrained.pth` checkpoint.
2. Fine-tune with **AsymmetricLoss**, **WeightedRandomSampler**, two-stage training.
3. Per-class **threshold tuning** on the validation set.
4. **TTA** — average over 8 augmented views.
5. Final test-set evaluation and **GradCAM** saliency maps.


In [1]:
import sys, time, copy
from pathlib import Path

_proj_root = (Path('.').resolve() if (Path('.') / 'experiments').exists()
              else Path('..').resolve())
_exp_dir   = _proj_root / 'experiments'

# Remove '' (cwd) and any stale project-root entry so experiments/utils.py wins
sys.path = [p for p in sys.path if p not in ('', '.', str(_proj_root))]
sys.path.insert(0, str(_proj_root))   # project root — for `eval`
sys.path.insert(0, str(_exp_dir))     # experiments  — for `utils` (highest priority)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import models as tv_models

from eval import LABEL_ORDER

from utils import (
    METRIC_KEYS, NORM_MEAN, NORM_STD,
    set_seed, load_dataset, split_dataset,
    TransformSubset, get_train_transform, get_eval_transform,
    compute_multilabel_metrics, print_metric_table,
    save_checkpoint, print_model_info,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, plot_per_class_examples,
    plot_training_history, denorm, labels_to_text,
)

SEED       = 42
SPLIT      = [0.7, 0.15, 0.15]
BASE_DIR   = str(_proj_root / 'data' / 'aggregated')
CKPT_DIR   = _proj_root / 'checkpoints'
NUM_LABELS = len(LABEL_ORDER)
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

set_seed(SEED)
print(f'Device: {DEVICE}  |  Labels: {NUM_LABELS}')
print(f'Label order: {LABEL_ORDER}')


Device: cuda  |  Labels: 12
Label order: ['pen', 'paper', 'book', 'clock', 'phone', 'laptop', 'chair', 'desk', 'bottle', 'keychain', 'backpack', 'calculator']


In [2]:
full_dataset = load_dataset(BASE_DIR)
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)
print(f'Train: {len(train_raw)}  |  Val: {len(val_raw)}  |  Test: {len(test_raw)}')


Train: 3180  |  Val: 681  |  Test: 682


In [3]:
def create_resnet50(num_labels):
    m = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Linear(m.fc.in_features, num_labels)
    return m

MODEL_REGISTRY = {
    'resnet50_pretrained': {
        'create_fn':  create_resnet50,
        'image_size': 224,
        'pretrained': True,
        'head_attr':  'fc',
        'gradcam_fn': lambda m: m.layer4[-1].conv3,
    },
}

BEST_ARCH     = 'resnet50_pretrained'
IMAGE_SIZE    = MODEL_REGISTRY[BEST_ARCH]['image_size']
IS_PRETRAINED = MODEL_REGISTRY[BEST_ARCH]['pretrained']
BATCH_SIZE    = 32

print(f'Architecture: {BEST_ARCH}  |  image_size={IMAGE_SIZE}')
print_model_info(create_resnet50(NUM_LABELS))

# Baseline metrics (before fine-tuning)
_eval_tf   = get_eval_transform(IMAGE_SIZE)
_ckpt_path = CKPT_DIR / f'final_{BEST_ARCH}.pth'
_m = create_resnet50(NUM_LABELS)
_m.load_state_dict(torch.load(_ckpt_path, map_location=DEVICE, weights_only=True))
_m.to(DEVICE).eval()
_val_l, _val_p, _val_pr = [], [], []
with torch.no_grad():
    for imgs, labels in DataLoader(TransformSubset(val_raw, _eval_tf), batch_size=64, num_workers=2):
        probs = torch.sigmoid(_m(imgs.to(DEVICE)))
        _val_l.append(labels)
        _val_p.append((probs >= 0.5).float().cpu())
        _val_pr.append(probs.cpu())
_base_val_m = compute_multilabel_metrics(torch.cat(_val_l), torch.cat(_val_p), probs=torch.cat(_val_pr))
df_cmp = pd.DataFrame([{**{k: round(_base_val_m[k], 4) for k in METRIC_KEYS}}], index=[BEST_ARCH])
print('\nBaseline val metrics (before fine-tuning):')
print(df_cmp[['f1_micro', 'exact_match', 'hamming_acc', 'mean_iou', 'precision_micro', 'recall_micro']].to_string())
del _m


Architecture: resnet50_pretrained  |  image_size=224
  Total params     :   23,532,620
  Trainable params :   23,532,620  (100.0%)
  Model size       : 89.77 MB  (float32 weights)


FileNotFoundError: [Errno 2] No such file or directory: '/home/abbasidaniyal/Projects/study_projects/MLE/project/checkpoints/final_resnet50_pretrained.pth'

## 1 · Advanced Fine-Tuning Setup

- **AsymmetricLoss (ASL)** — down-weights easy negatives.
- **WeightedRandomSampler** — up-samples rare label combinations.
- **freeze_backbone / unfreeze_all** — for two-stage training.


In [ ]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05):
        super().__init__()
        self.gamma_neg, self.gamma_pos, self.clip = gamma_neg, gamma_pos, clip

    def forward(self, logits, targets):
        probs     = torch.sigmoid(logits)
        probs_neg = (probs - self.clip).clamp(min=0.0)
        loss_pos  = -targets       * (1 - probs)    ** self.gamma_pos * torch.log(probs.clamp(1e-8))
        loss_neg  = -(1 - targets) * probs_neg      ** self.gamma_neg * torch.log((1 - probs_neg).clamp(1e-8))
        return (loss_pos + loss_neg).mean()


def make_weighted_sampler(train_raw):
    print('Computing label frequencies for weighted sampler...')
    all_labels = torch.stack([train_raw[i][1] for i in range(len(train_raw))])
    pos_freq   = all_labels.float().mean(dim=0).clamp(min=1e-6)
    weights    = []
    for i in range(len(all_labels)):
        active = all_labels[i].bool()
        weights.append((1.0 / pos_freq[active].mean().item()) if active.any() else 1.0)
    weights = torch.tensor(weights, dtype=torch.float)
    print(f'  weight range: {weights.min():.2f} - {weights.max():.2f}')
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


def get_param_groups(model, arch):
    head_mod = getattr(model, MODEL_REGISTRY[arch]['head_attr'])
    head_ids = {id(p) for p in head_mod.parameters()}
    return [p for p in model.parameters() if id(p) not in head_ids], list(head_mod.parameters())

def freeze_backbone(model, arch):
    for p in get_param_groups(model, arch)[0]: p.requires_grad = False

def unfreeze_all(model):
    for p in model.parameters(): p.requires_grad = True

print('ASL, sampler, and parameter helpers ready.')


## 2 · Two-Stage Fine-Tuning

| Stage | Trainable | LR | Epochs | Purpose |
|-------|-----------|-----|--------|---------|
| A | head only | 1e-3 | up to 10 | Adapt classifier to ASL + weighted data |
| B | full model | backbone 1e-5 / head 1e-4 | up to 30 | End-to-end refinement |


In [ ]:
def _run_finetune(model, train_loader, val_loader, optimizer, criterion, device,
                  max_epochs, warmup_epochs=2, early_stop_patience=5,
                  lr_reduce_patience=3, grad_clip=1.0, threshold=0.5, stage_label=''):
    base_lrs  = [pg['lr'] for pg in optimizer.param_groups]
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=lr_reduce_patience)
    history   = {k: {'train': [], 'val': []} for k in METRIC_KEYS}
    best_val_f1, best_state, no_improve = -1.0, None, 0

    for epoch in range(1, max_epochs + 1):
        if epoch <= warmup_epochs:
            scale = epoch / warmup_epochs
            for pg, base_lr in zip(optimizer.param_groups, base_lrs): pg['lr'] = base_lr * scale

        model.train()
        tr_l, tr_p, tr_pr, tr_lg = [], [], [], []
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            logits = model(images)
            criterion(logits, targets).backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            with torch.no_grad(): probs = torch.sigmoid(logits)
            tr_l.append(targets.cpu()); tr_p.append((probs >= threshold).float().cpu())
            tr_pr.append(probs.cpu()); tr_lg.append(logits.detach().cpu())
        train_m = compute_multilabel_metrics(torch.cat(tr_l), torch.cat(tr_p),
                                             probs=torch.cat(tr_pr), logits=torch.cat(tr_lg))

        model.eval()
        val_l, val_p, val_pr, val_lg = [], [], [], []
        with torch.no_grad():
            for images, labels in val_loader:
                logits = model(images.to(device)); probs = torch.sigmoid(logits)
                val_l.append(labels.cpu()); val_p.append((probs >= threshold).float().cpu())
                val_pr.append(probs.cpu()); val_lg.append(logits.cpu())
        val_m = compute_multilabel_metrics(torch.cat(val_l), torch.cat(val_p),
                                           probs=torch.cat(val_pr), logits=torch.cat(val_lg))

        for k in METRIC_KEYS:
            history[k]['train'].append(train_m[k])
            history[k]['val'].append(val_m[k])

        val_f1 = val_m['f1_micro']
        if epoch > warmup_epochs: scheduler.step(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1, best_state, no_improve = val_f1, copy.deepcopy(model.state_dict()), 0
        else: no_improve += 1

        lr_str = '/'.join(f"{pg['lr']:.1e}" for pg in optimizer.param_groups)
        print(f'[{stage_label}] Epoch {epoch:>2}/{max_epochs}  lr={lr_str}  '
              f'train_F1={train_m["f1_micro"]:.4f}  val_F1={val_f1:.4f}' + ('  *' if no_improve == 0 else ''))
        if no_improve >= early_stop_patience:
            print(f'  [Early stop] No improvement for {early_stop_patience} epochs.')
            break

    model.load_state_dict(best_state)
    return best_state, best_val_f1, history, epoch

print('Fine-tuning loop defined.')


In [ ]:
ckpt_path  = CKPT_DIR / f'final_{BEST_ARCH}.pth'
best_model = create_resnet50(NUM_LABELS)
best_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
best_model.to(DEVICE)
print(f'Loaded: {ckpt_path}')

train_tf = get_train_transform(IMAGE_SIZE)
eval_tf  = get_eval_transform(IMAGE_SIZE)
sampler  = make_weighted_sampler(train_raw)

train_loader_ft = DataLoader(TransformSubset(train_raw, train_tf),
                             batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader_ft   = DataLoader(TransformSubset(val_raw, eval_tf),
                             batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
asl = AsymmetricLoss(gamma_neg=4.0, gamma_pos=1.0, clip=0.05)
print('Data loaders and ASL ready.')


In [ ]:
t0 = time.time()
print('=== Stage A: Head-only fine-tuning ===')
freeze_backbone(best_model, BEST_ARCH)
_, head_params = get_param_groups(best_model, BEST_ARCH)
print(f'  Trainable: {sum(p.numel() for p in best_model.parameters() if p.requires_grad):,}')
opt_a = optim.AdamW(head_params, lr=1e-3, weight_decay=1e-4)
state_a, f1_a, hist_a, ep_a = _run_finetune(
    best_model, train_loader_ft, val_loader_ft, opt_a, asl, DEVICE,
    max_epochs=10, warmup_epochs=2, early_stop_patience=4, stage_label='Stage-A',
)
print(f'\nStage A done - best val F1: {f1_a:.4f}  ({ep_a} epochs, {time.time()-t0:.0f}s)')
plot_training_history(hist_a, ep_a, experiment_name=f'{BEST_ARCH} Stage-A', lr=1e-3, weight_decay=1e-4)


In [ ]:
print('=== Stage B: Full-model fine-tuning ===')
unfreeze_all(best_model)
print(f'  Trainable: {sum(p.numel() for p in best_model.parameters()):,}')
backbone_params, head_params = get_param_groups(best_model, BEST_ARCH)
opt_b = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},
    {'params': head_params,     'lr': 1e-4},
], weight_decay=1e-4)
t0 = time.time()
state_b, f1_b, hist_b, ep_b = _run_finetune(
    best_model, train_loader_ft, val_loader_ft, opt_b, asl, DEVICE,
    max_epochs=30, warmup_epochs=3, early_stop_patience=7, stage_label='Stage-B',
)
print(f'\nStage B done - best val F1: {f1_b:.4f}  ({ep_b} epochs, {time.time()-t0:.0f}s)')
plot_training_history(hist_b, ep_b, experiment_name=f'{BEST_ARCH} Stage-B', lr=1e-5, weight_decay=1e-4)


## 3 · Per-Class Threshold Tuning

Grid-search $t_c \in [0.10, 0.85]$ per label on the **validation** set to maximise per-class F1.


In [ ]:
best_model.eval()
val_all_labels, val_all_probs = [], []
with torch.no_grad():
    for imgs, labels in val_loader_ft:
        probs = torch.sigmoid(best_model(imgs.to(DEVICE)))
        val_all_labels.append(labels); val_all_probs.append(probs.cpu())
val_labels = torch.cat(val_all_labels)
val_probs  = torch.cat(val_all_probs)

THRESHOLD_GRID  = torch.arange(0.10, 0.90, 0.05).tolist()
best_thresholds = torch.full((NUM_LABELS,), 0.5)

print(f'{"Label":<14} {"t_tuned":>8} {"F1@0.5":>8} {"F1@tuned":>10}')
print('-' * 44)
for c, label in enumerate(LABEL_ORDER):
    best_f1, best_t, f1_default = 0.0, 0.5, 0.0
    for t in THRESHOLD_GRID:
        preds_c = (val_probs[:, c] >= t).float()
        tp = ((preds_c == 1) & (val_labels[:, c] == 1)).sum().float()
        fp = ((preds_c == 1) & (val_labels[:, c] == 0)).sum().float()
        fn = ((preds_c == 0) & (val_labels[:, c] == 1)).sum().float()
        f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).item()
        if abs(t - 0.5) < 0.01: f1_default = f1
        if f1 > best_f1: best_f1, best_t = f1, t
    best_thresholds[c] = best_t
    print(f'{label:<14} {best_t:>8.2f} {f1_default:>8.4f} {best_f1:>10.4f}')
print(f'\nTuned thresholds: {[round(t.item(), 2) for t in best_thresholds]}')


## 4 · Test-Time Augmentation (TTA)

8 random augmented views + 1 deterministic view, averaged before applying tuned thresholds.


In [ ]:
TTA_N_AUG = 8
tta_tf  = get_train_transform(IMAGE_SIZE)
eval_tf = get_eval_transform(IMAGE_SIZE)

def evaluate_tta(model, raw_subset, n_aug, threshold_vec, device):
    model.eval()
    all_labels, all_probs = [], []
    for i in range(len(raw_subset)):
        pil_img, label = raw_subset[i]
        views = [eval_tf(pil_img)] + [tta_tf(pil_img) for _ in range(n_aug - 1)]
        with torch.no_grad():
            avg_probs = torch.sigmoid(model(torch.stack(views).to(device))).mean(dim=0).cpu()
        all_labels.append(label); all_probs.append(avg_probs)
    all_labels = torch.stack(all_labels)
    all_probs  = torch.stack(all_probs)
    m_tuned   = compute_multilabel_metrics(all_labels, (all_probs >= threshold_vec.unsqueeze(0)).float(), probs=all_probs)
    m_default = compute_multilabel_metrics(all_labels, (all_probs >= 0.5).float(), probs=all_probs)
    return m_tuned, m_default, all_labels, (all_probs >= threshold_vec.unsqueeze(0)).float(), all_probs

print(f'Running TTA ({TTA_N_AUG} views) on {len(test_raw)} test images...')
t0 = time.time()
m_tta_tuned, m_tta_default, test_labels, test_preds_tuned, test_probs = \
    evaluate_tta(best_model, test_raw, TTA_N_AUG, best_thresholds, DEVICE)
print(f'TTA done in {time.time()-t0:.1f}s')

# Standard eval for comparison
test_loader_eval = DataLoader(TransformSubset(test_raw, eval_tf),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
best_model.eval()
std_l, std_pr, std_lg = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader_eval:
        logits = best_model(imgs.to(DEVICE))
        std_l.append(labels); std_pr.append(torch.sigmoid(logits).cpu()); std_lg.append(logits.cpu())
std_labels, std_probs, std_logits = torch.cat(std_l), torch.cat(std_pr), torch.cat(std_lg)
m_std_05    = compute_multilabel_metrics(std_labels, (std_probs >= 0.5).float(), probs=std_probs, logits=std_logits)
m_std_tuned = compute_multilabel_metrics(std_labels, (std_probs >= best_thresholds.unsqueeze(0)).float(),
                                         probs=std_probs, logits=std_logits)

df_variants = pd.DataFrame([
    {'config': 'baseline (no TTA, t=0.5)',  **{k: round(m_std_05[k],     4) for k in METRIC_KEYS}},
    {'config': 'tuned thresholds, no TTA',  **{k: round(m_std_tuned[k],  4) for k in METRIC_KEYS}},
    {'config': 'TTA, t=0.5',                **{k: round(m_tta_default[k],4) for k in METRIC_KEYS}},
    {'config': 'TTA + tuned thresholds',    **{k: round(m_tta_tuned[k],  4) for k in METRIC_KEYS}},
]).set_index('config')
print('\n=== Test-set evaluation variants ===')
print(df_variants[['f1_micro','exact_match','hamming_acc','mean_iou','precision_micro','recall_micro','loss']].to_string())


## 5 · Detailed Test-Set Analysis


In [ ]:
final_labels = test_labels
final_preds  = test_preds_tuned
final_probs  = test_probs

plot_per_class_metrics(final_labels, final_preds)
plot_confusion_matrices(final_labels, final_preds)

_imgs_norm = torch.stack([eval_tf(test_raw[i][0]) for i in range(len(test_raw))])
correct_idx, partial_idx, incorrect_idx = categorize_predictions(final_labels, final_preds)
show_prediction_examples(correct_idx,   _imgs_norm, final_labels, final_preds, title='Correct predictions',   n=6)
show_prediction_examples(partial_idx,   _imgs_norm, final_labels, final_preds, title='Partial predictions',   n=6)
show_prediction_examples(incorrect_idx, _imgs_norm, final_labels, final_preds, title='Incorrect predictions', n=4)


## 6 · GradCAM Saliency Maps

One representative image per class with GradCAM overlay (ResNet-50 `layer4[-1].conv3`).


In [ ]:
import torch.nn.functional as F

class GradCAM:
    def __init__(self, model, target_layer):
        self.model, self._act, self._grad = model, None, None
        self._fwd = target_layer.register_forward_hook(
            lambda m, i, o: self.__setattr__('_act', o.detach()))
        self._bwd = target_layer.register_full_backward_hook(
            lambda m, gi, go: self.__setattr__('_grad', go[0].detach()))

    def __call__(self, img_tensor, device):
        self.model.eval()
        x = img_tensor.unsqueeze(0).to(device).requires_grad_(True)
        self.model.zero_grad()
        torch.sigmoid(self.model(x)).sum().backward()
        weights = self._grad.mean(dim=[2, 3], keepdim=True)
        cam = F.relu((weights * self._act).sum(dim=1, keepdim=True)).squeeze().cpu()
        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        return cam.numpy()

    def remove(self): self._fwd.remove(); self._bwd.remove()


per_class_imgs = {}
for pil_img, target in train_raw:
    for i, label in enumerate(LABEL_ORDER):
        if target.int()[i] == 1 and label not in per_class_imgs:
            per_class_imgs[label] = (pil_img.copy(), target.clone())
    if len(per_class_imgs) == NUM_LABELS: break

target_layer = MODEL_REGISTRY[BEST_ARCH]['gradcam_fn'](best_model)
cam_fn = GradCAM(best_model, target_layer)
print(f'GradCAM layer: {type(target_layer).__name__}')

n_cols = len(LABEL_ORDER)
fig, axes = plt.subplots(2, n_cols, figsize=(2.8 * n_cols, 6))
for col, label in enumerate(LABEL_ORDER):
    if label not in per_class_imgs:
        axes[0][col].axis('off'); axes[1][col].axis('off'); continue
    pil_img, _ = per_class_imgs[label]
    img_t   = eval_tf(pil_img)
    img_np  = denorm(img_t).permute(1, 2, 0).numpy()
    h, w    = img_t.shape[1], img_t.shape[2]
    hmap    = cam_fn(img_t, DEVICE)
    hmap_up = F.interpolate(torch.tensor(hmap).unsqueeze(0).unsqueeze(0).float(),
                            size=(h, w), mode='bilinear', align_corners=False).squeeze().numpy()
    axes[0][col].imshow(img_np); axes[0][col].set_title(label, fontsize=8, pad=3); axes[0][col].axis('off')
    axes[1][col].imshow(img_np); axes[1][col].imshow(hmap_up, cmap='jet', alpha=0.45); axes[1][col].axis('off')

axes[0][0].set_ylabel('Original', fontsize=9)
axes[1][0].set_ylabel('GradCAM',  fontsize=9)
plt.suptitle(f'GradCAM - one example per class ({BEST_ARCH})', fontsize=11, y=1.01)
plt.tight_layout(pad=0.3, h_pad=0.5)
plt.show()
cam_fn.remove()


## 7 · Save & Summary


In [ ]:
OUT_PATH = CKPT_DIR / f'final_finetuned_{BEST_ARCH}.pth'
save_checkpoint(best_model.state_dict(), OUT_PATH)

print('\n' + '=' * 60)
print(f'  Architecture      : {BEST_ARCH}')
print(f'  Image size        : {IMAGE_SIZE}')
print(f'  Criterion         : AsymmetricLoss (gamma_neg=4, gamma_pos=1, clip=0.05)')
print(f'  Sampler           : WeightedRandomSampler')
print(f'  Fine-tuning       : Stage A (head) + Stage B (full)')
print(f'  TTA views         : {TTA_N_AUG}')
print(f'  Threshold tuning  : per-class (val F1)')
print(f'  Tuned thresholds  : {[round(t.item(), 2) for t in best_thresholds]}')
print('-' * 60)
print(f'  {"Metric":<20} {"Before FT":>10} {"After FT (TTA+tuned)":>22}')
print(f'  {"-"*54}')
for k in METRIC_KEYS:
    before = df_cmp.loc[BEST_ARCH, k] if k in df_cmp.columns else float('nan')
    after  = m_tta_tuned[k]
    print(f'  {k:<20} {before:>10.4f} {after:>22.4f}  ({after-before:+.4f})')
print('=' * 60)
print(f'\nCheckpoint saved: {OUT_PATH}')
